In [1]:
# Cell 1 — Imports
import sys, importlib.util, json
from pathlib import Path
from datetime import datetime, timedelta
from math import sqrt
import numpy as np
import pandas as pd
import cv2
import toml
import msgpack
import msgpack_numpy as m
m.patch()                        # makes msgpack decode numpy arrays natively
# import sys
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.interpolate import interp1d
from scipy.signal import correlate

print("Imports OK ✓")


Imports OK ✓


In [3]:
# Cell 2 — Recording-specific paths
# ── UPDATE THESE FOR EACH RECORDING ──────────────────────────────────────────
RECORDING_DIR = Path("E:\\data\\noark_data_may_6")

FRAMES_MSGPACK    = RECORDING_DIR / "linear_t0_slow_may_6" / "webcam_color.msgpack"
TIMESTAMPS_MSGPACK= RECORDING_DIR /"linear_t0_slow_may_6"/ "webcam_timestamp.msgpack"
ENC_CSV           = RECORDING_DIR /"linear_t0_slow_may_6"/ "encoder_data.csv"
# "E:\data\noark_data_may_6\linear_t0_slow_may_6\webcam_timestamp.msgpack"
# Calibration paths
repo_root        = Path("E:/Ragav/MS Bio Engineering/NOARK_backbone")
CAM_CALIB_PATH   = repo_root / "notebooks" / "calibration" / "output" / "good.toml"
TABLE_CALIB_PATH = repo_root / "estimator" / "charuco_pose" / "charuco_pose.toml"

# Verify files exist
for p in [FRAMES_MSGPACK, TIMESTAMPS_MSGPACK, ENC_CSV, CAM_CALIB_PATH, TABLE_CALIB_PATH]:
    status = "✓" if p.exists() else "✗ MISSING"
    print(f"{status}  {p}")
    # Quick check — run this first to see timestamp msgpack structure
with open(TIMESTAMPS_MSGPACK, "rb") as f:
    unpacker = msgpack.Unpacker(f, raw=False)
    samples = [next(unpacker) for _ in range(5)]
for s in samples:
    print(s)


✓  E:\data\noark_data_may_6\linear_t0_slow_may_6\webcam_color.msgpack
✓  E:\data\noark_data_may_6\linear_t0_slow_may_6\webcam_timestamp.msgpack
✓  E:\data\noark_data_may_6\linear_t0_slow_may_6\encoder_data.csv
✓  E:\Ragav\MS Bio Engineering\NOARK_backbone\notebooks\calibration\output\good.toml
✓  E:\Ragav\MS Bio Engineering\NOARK_backbone\estimator\charuco_pose\charuco_pose.toml
[0, '2026-05-06 16:07:11.429239']
[0, '2026-05-06 16:07:11.441026']
[0, '2026-05-06 16:07:11.455024']
[0, '2026-05-06 16:07:11.465106']
[0, '2026-05-06 16:07:11.471013']


In [ ]:
# Cell 2b — TABLE FRAME CALIBRATION (run once per setup)
# Mirrors charuco_estimator.py exactly — same board, same dist coeffs shape,
# same toml structure. Run once, then Cell 4 loads the saved toml.

TABLE_FRAMES_MSGPACK = Path("E:\\data\\noark_data_may_6") / "table_frame_recording" / "webcam_color.msgpack"
TABLE_CALIB_OUTPUT   = repo_root / "estimator" / "charuco_pose" / "charuco_pose.toml"

# ── Load table frames ─────────────────────────────────────────────────────────
print("Loading table recording frames...")
with open(TABLE_FRAMES_MSGPACK, "rb") as f:
    unpacker = msgpack.Unpacker(f, raw=False)
    table_frames = [item for item in unpacker]
print(f"Table frames loaded: {len(table_frames)}")

# ── Process frames — mirrors charuco_estimator.py::process_frame() ────────────
rvecs_collected = []
tvecs_collected = []
n_failed = 0

for i, frame_raw in enumerate(table_frames):
    # RGB msgpack frame → grayscale (mirrors YUV420 Y-channel extraction)
    frame  = np.array(frame_raw, dtype=np.uint8)
    gray   = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)

    # Flip horizontally — matches charuco_estimator.py line: gray_image = cv2.flip(gray_image, 1)
    gray   = cv2.flip(gray, 1)

    # Undistort — same as _init_undistortion_maps + remap in charuco_estimator.py
    undist = cv2.remap(gray, map1, map2,
                       interpolation=cv2.INTER_LINEAR,
                       borderMode=cv2.BORDER_CONSTANT)

    # Detect ChArUco corners — same as charuco_estimator.py
    charuco_corners, charuco_ids, _, _ = charuco_detector.detectBoard(undist)

    if charuco_corners is None or charuco_ids is None or len(charuco_ids) < 4:
        n_failed += 1
        continue

    # Estimate pose — same call as charuco_estimator.py (np.zeros((4,1)) matches original)
    ok, rvec, tvec = cv2.aruco.estimatePoseCharucoBoard(
        charuco_corners, charuco_ids, board,
        new_K,
        np.zeros((4, 1)),   # no dist — image already undistorted, matches charuco_estimator.py
        None, None
    )
    if ok:
        rvecs_collected.append(rvec.flatten())
        tvecs_collected.append(tvec.flatten())
        if i % 50 == 0:
            print(f"  Frame {i}: tvec={tvec.ravel().round(4)}")

print(f"\nFrames with valid pose : {len(rvecs_collected)} / {len(table_frames)}")
print(f"Frames failed          : {n_failed}")
assert len(rvecs_collected) > 0, "No valid ChArUco pose found — check board params and lighting"

# ── Average across all valid frames for stability ─────────────────────────────
mean_rvec = np.mean(rvecs_collected, axis=0)
mean_tvec = np.mean(tvecs_collected, axis=0)
std_rvec  = np.std(rvecs_collected,  axis=0)
std_tvec  = np.std(tvecs_collected,  axis=0)
R_cal, _  = cv2.Rodrigues(mean_rvec)

print(f"\nMean tvec : {mean_tvec.round(5)}  (std: {std_tvec.round(5)})")
print(f"Rotation matrix:\n{np.round(R_cal, 4)}")
print(f"det(R) = {np.linalg.det(R_cal):.6f}  ({'OK' if abs(np.linalg.det(R_cal)-1.0)<1e-4 else 'BAD'})")

# ── Save toml — same structure as charuco_estimator.py::save_calibration_data() ──
TABLE_CALIB_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
calib_out = {
    "camera_matrix"  : cam_K.tolist(),         # same as charuco_estimator.py
    "dist_coeffs"    : cam_dist.tolist(),       # same as charuco_estimator.py
    "rvec"           : mean_rvec.reshape(-1, 1).tolist(),   # same shape as cv2 output
    "tvec"           : mean_tvec.reshape(-1, 1).tolist(),   # same shape as cv2 output
    "rotation_matrix": R_cal.tolist(),
    "n_frames_used"  : len(rvecs_collected),
    "rvec_std"       : std_rvec.tolist(),
    "tvec_std"       : std_tvec.tolist(),
}
with open(TABLE_CALIB_OUTPUT, "w") as f:
    toml.dump(calib_out, f)
print(f"\nSaved -> {TABLE_CALIB_OUTPUT}")
print("Now run Cell 4 to load R_table / T_table.")

In [ ]:
# Cell 3 — Kinematics constants + marker config
# ── UPDATE PULLEY POSITIONS FROM YOUR NOTES SHEET ────────────────────────────
P2 = np.array([-0.495, 0, -0.027])   # Left upper pulley  (from nkin.py)
P4 = np.array([ 0.495, 0, -0.027])   # Right upper pulley (from nkin.py)
r_ML = r_MR = 0.033                  # Motor spool radius (m)

# ArUco marker IDs on NOARK and their offsets in NOARK body frame (metres)
MARKER_IDS     = [12, 14, 20]
MARKER_LENGTH  = 0.049   # metres
MARKER_OFFSETS = {
    12: np.array([ 0.000, 0, -0.055]),
    14: np.array([-0.126, 0, -0.054]),
    20: np.array([ 0.126, 0, -0.054]),
}

print("Config loaded ✓")
print(f"P2 = {P2*100} cm")
print(f"P4 = {P4*100} cm")


In [ ]:
# Cell 4 — Load camera + table calibrations
calib     = toml.load(CAM_CALIB_PATH)
cam_K     = np.array(calib["calibration"]["camera_matrix"])
cam_dist  = np.array(calib["calibration"]["dist_coeffs"])

tbl       = toml.load(TABLE_CALIB_PATH)
R_table   = np.array(tbl["rotation_matrix"]).reshape(3, 3)
T_table   = np.array(tbl["tvec"]).reshape(3, 1)

# Compute undistortion maps once (fisheye)
resolution = tuple(calib["camera"]["resolution"])
new_K = cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(
    cam_K, cam_dist, resolution, np.eye(3), balance=1.0
)
map1, map2 = cv2.fisheye.initUndistortRectifyMap(
    cam_K, cam_dist, np.eye(3), new_K, resolution, cv2.CV_16SC2
)

print(f"Camera resolution : {resolution}")
print(f"R_table :\n{np.round(R_table, 4)}")
print(f"T_table  : {T_table.flatten().round(4)} m")
print("Calibrations loaded ✓")


In [ ]:
# Cell 5 — Load rawframes and timestamps from msgpack
print("Loading timestamps...")
with open(TIMESTAMPS_MSGPACK, "rb") as f:
    unpacker = msgpack.Unpacker(f, raw=False)
    raw_ts = [item for item in unpacker]

print(f"Timestamps loaded : {len(raw_ts)}")
print(f"First entry type  : {type(raw_ts[0])}  value: {raw_ts[0]}")

# Convert to datetime
if isinstance(raw_ts[0], (int, float)):
    if raw_ts[0] > 1e10:
        timestamps = [datetime.fromtimestamp(t / 1000.0) for t in raw_ts]
    else:
        timestamps = [datetime.fromtimestamp(t) for t in raw_ts]
elif isinstance(raw_ts[0], str):
    timestamps = [datetime.fromisoformat(t) for t in raw_ts]
elif isinstance(raw_ts[0], list):
    # Entry format is [frame_id, datetime_string]
    timestamps = [datetime.strptime(t[1], "%Y-%m-%d %H:%M:%S.%f") for t in raw_ts]
else:
    timestamps = list(raw_ts)

print(f"First timestamp   : {timestamps[0]}")
print(f"Last  timestamp   : {timestamps[-1]}")
duration = (timestamps[-1] - timestamps[0]).total_seconds()
print(f"Duration          : {duration:.2f} s")
print(f"Mean FPS          : {len(timestamps)/duration:.1f}")

print("\nLoading raw frames...")
with open(FRAMES_MSGPACK, "rb") as f:
    unpacker = msgpack.Unpacker(f, raw=False)
    raw_frames = [item for item in unpacker]

print(f"Frames loaded     : {len(raw_frames)}")
assert len(raw_frames) == len(timestamps), (
    f"Frame/timestamp count mismatch: {len(raw_frames)} vs {len(timestamps)}")

frame0 = np.array(raw_frames[0], dtype=np.uint8)
print(f"Frame shape       : {frame0.shape}  dtype={frame0.dtype}")

In [ ]:
# Cell 6 — ChArUco detector setup (mirrors charuco_estimator.py)
dictionary = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
board      = cv2.aruco.CharucoBoard((4, 3), 0.037, 0.027, dictionary)
charuco_detector = cv2.aruco.CharucoDetector(board)
aruco_detector   = cv2.aruco.ArucoDetector(
    dictionary, cv2.aruco.DetectorParameters()
)
print("ChArUco detector ready ✓")

def estimate_markers_in_frame(frame_rgb):
    """
    Given one RGB frame (H,W,3):
    1. Convert to grayscale
    2. Undistort (fisheye)
    3. Detect all ArUco markers
    4. For each of MARKER_IDS: estimate pose via solvePnP
    Returns dict {marker_id: (rvec, tvec)} for detected markers.
    """
    gray = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2GRAY)
    undist = cv2.remap(gray, map1, map2,
                       interpolation=cv2.INTER_LINEAR,
                       borderMode=cv2.BORDER_CONSTANT)

    corners, ids, _ = aruco_detector.detectMarkers(undist)
    if ids is None or len(ids) == 0:
        return {}

    ids_flat = ids.flatten().tolist()
    result = {}
    half = MARKER_LENGTH / 2.0
    obj_pts = np.array([[-half, half, 0],
                        [ half, half, 0],
                        [ half,-half, 0],
                        [-half,-half, 0]], dtype=np.float32)

    for corner, mid_arr in zip(corners, ids):
        mid = int(mid_arr[0])
        if mid not in MARKER_IDS:
            continue
        img_pts = corner[0].astype(np.float32)
        ok, rvec, tvec = cv2.solvePnP(
            obj_pts, img_pts, new_K, np.zeros((4,1))
        )
        if ok:
            result[mid] = (rvec.flatten(), tvec.flatten())
    return result


In [ ]:
# Cell 7 — Process all frames → extract NOARK position in table frame
# This replicates camera_pose.py get_centroid + get_local_coordinates

def get_centroid_from_detections(detections):
    """
    detections: dict {marker_id: (rvec, tvec)}
    Returns centroid in CAMERA frame as (3,) or None.
    """
    points = []
    for mid in MARKER_IDS:
        if mid not in detections:
            continue
        rvec, tvec = detections[mid]
        R_m, _ = cv2.Rodrigues(rvec)
        offset  = MARKER_OFFSETS[mid].reshape(3, 1)
        pos_cam = R_m @ offset + tvec.reshape(3, 1)
        points.append(pos_cam.flatten())
    if not points:
        return None
    return np.mean(points, axis=0)

def cam_to_table(p_cam):
    """Transform point from camera frame to table frame."""
    return (R_table.T @ (p_cam.reshape(3,1) - T_table)).flatten()

# ── Process every frame ───────────────────────────────────────────────────────
print(f"Processing {len(raw_frames)} frames...")
rows = []
for i, (frame_raw, ts) in enumerate(zip(raw_frames, timestamps)):
    if i % 500 == 0:
        print(f"  Frame {i}/{len(raw_frames)}  ({100*i/len(raw_frames):.0f}%)")

    frame = np.array(frame_raw, dtype=np.uint8)
    detections = estimate_markers_in_frame(frame)
    centroid   = get_centroid_from_detections(detections)

    if centroid is not None:
        pos_table = cam_to_table(centroid)
        x_cam, y_cam, z_cam = pos_table
    else:
        x_cam = y_cam = z_cam = np.nan

    rows.append({
        "timestamp": ts,
        "x_cam":     x_cam,
        "y_cam":     y_cam,
        "z_cam":     z_cam,
        "n_markers": len(detections),
    })

cam_df = pd.DataFrame(rows)
valid  = cam_df["x_cam"].notna().sum()
print(f"\nDone. Valid frames: {valid} / {len(cam_df)}  ({100*valid/len(cam_df):.1f}%)")
print(f"X range: {cam_df['x_cam'].dropna().min()*100:.2f} → {cam_df['x_cam'].dropna().max()*100:.2f} cm")
print(f"Z range: {cam_df['z_cam'].dropna().min()*100:.2f} → {cam_df['z_cam'].dropna().max()*100:.2f} cm")
cam_df.head(5)


In [ ]:
# Cell 8 — Save processed camera positions to CSV
out_csv = RECORDING_DIR / "camera_data_processed.csv"
cam_df.to_csv(out_csv, index=False)
print(f"Saved → {out_csv}")
print(f"Shape  : {cam_df.shape}")


In [ ]:
# Cell 9 — Sync pin detection + time alignment
# The sync pin column tells us the hardware trigger moment.
# If your msgpack timestamps already start at the sync moment, set SYNC_FROM_ENCODER=True
SYNC_FROM_ENCODER = True   # ← set False if cam timestamps have a sync_pin column

enc_df = pd.read_csv(ENC_CSV, parse_dates=["timestamp"])
enc_df["timestamp"] = pd.to_datetime(enc_df["timestamp"])
cam_df["timestamp"] = pd.to_datetime(cam_df["timestamp"])

if SYNC_FROM_ENCODER:
    # Find sync rising edge in encoder CSV
    assert "sync_pin" in enc_df.columns, f"sync_pin not found in {ENC_CSV.name}"
    sync_rows = enc_df[enc_df["sync_pin"] == 1]
    assert len(sync_rows) > 0, "sync_pin never goes HIGH"
    sync_timestamp = sync_rows.iloc[0]["timestamp"]
    print(f"Sync pin detected in encoder CSV at: {sync_timestamp}")
else:
    # Use first camera frame as t0
    sync_timestamp = cam_df["timestamp"].iloc[0]
    print(f"Using first camera frame as t0: {sync_timestamp}")

t0 = sync_timestamp
cam_df["t_sec"] = (cam_df["timestamp"] - t0).dt.total_seconds()
enc_df["t_sec"] = (enc_df["timestamp"] - t0).dt.total_seconds()

cam_sync = cam_df[cam_df["t_sec"] >= 0].reset_index(drop=True)
enc_s    = enc_df.sort_values("t_sec").reset_index(drop=True)

print(f"Camera frames from sync onward : {len(cam_sync)}")
print(f"Encoder frames                 : {len(enc_s)}")
print(f"Camera t range : {cam_sync['t_sec'].min():.3f} → {cam_sync['t_sec'].max():.3f} s")
print(f"Encoder t range: {enc_s['t_sec'].min():.3f} → {enc_s['t_sec'].max():.3f} s")


In [ ]:
# Cell 10 — Init cable lengths + encoder kinematics (mirrors nkin.py)
from math import sqrt

# Init position from first valid camera frame after sync
first_valid = cam_sync[cam_sync["x_cam"].notna()].iloc[0]
N0 = np.array([first_valid["x_cam"], 0, first_valid["z_cam"]])

P2_to_NOARK_init = sqrt((N0[0]-P2[0])**2 + (N0[2]-P2[2])**2)
P4_to_NOARK_init = sqrt((N0[0]-P4[0])**2 + (N0[2]-P4[2])**2)

print(f"Init position (table frame): x={N0[0]*100:.2f} cm  z={N0[2]*100:.2f} cm")
print(f"P2_to_NOARK_init : {P2_to_NOARK_init*100:.2f} cm")
print(f"P4_to_NOARK_init : {P4_to_NOARK_init*100:.2f} cm")

def circle_intersect_xz(p2, p4, r_l, r_r):
    x2, z2 = p2[0], p2[2]
    x4, z4 = p4[0], p4[2]
    d = sqrt((x4-x2)**2 + (z4-z2)**2)
    if d > r_l+r_r or d < abs(r_l-r_r) or d == 0:
        return np.nan, np.nan
    a  = (r_l**2 - r_r**2 + d**2) / (2*d)
    h2 = r_l**2 - a**2
    if h2 < 0:
        return np.nan, np.nan
    h  = sqrt(h2)
    xm = x2 + a*(x4-x2)/d
    zm = z2 + a*(z4-z2)/d
    return xm + h*(z4-z2)/d, zm - h*(x4-x2)/d

x_enc_list, z_enc_list = [], []
for _, row in enc_s.iterrows():
    delta_L  = r_ML * np.deg2rad(float(row["enc1"]))
    delta_R  = r_MR * np.deg2rad(float(row["enc2"]))
    L_free_L = P2_to_NOARK_init + delta_L
    L_free_R = P4_to_NOARK_init - delta_R
    xe, ze   = circle_intersect_xz(P2, P4, L_free_L, L_free_R)
    x_enc_list.append(xe)
    z_enc_list.append(ze)

enc_s["x_enc"] = x_enc_list
enc_s["z_enc"] = z_enc_list

valid_enc = enc_s["x_enc"].notna().sum()
print(f"\nValid encoder frames: {valid_enc} / {len(enc_s)}")
enc_s[["timestamp","enc1","enc2","x_enc","z_enc"]].head(5).round(4)


In [ ]:
# Cell 11 — Merge camera + encoder onto common time grid
t_common = np.union1d(cam_sync["t_sec"].values, enc_s["t_sec"].values)
t_common  = np.sort(t_common)
merged    = pd.DataFrame({"t_sec": t_common})

# Camera — convert metres → cm
cam_valid = cam_sync[cam_sync["x_cam"].notna()]
interp_cx = interp1d(cam_valid["t_sec"], cam_valid["x_cam"]*100,
                     kind="linear", bounds_error=False, fill_value=np.nan)
interp_cz = interp1d(cam_valid["t_sec"], cam_valid["z_cam"]*100,
                     kind="linear", bounds_error=False, fill_value=np.nan)
merged["x_cam_cm"] = interp_cx(merged["t_sec"])
merged["z_cam_cm"] = interp_cz(merged["t_sec"])

# Encoder — convert metres → cm
enc_valid = enc_s[enc_s["x_enc"].notna()]
interp_ex = interp1d(enc_valid["t_sec"], enc_valid["x_enc"]*100,
                     kind="linear", bounds_error=False, fill_value=np.nan)
interp_ez = interp1d(enc_valid["t_sec"], enc_valid["z_enc"]*100,
                     kind="linear", bounds_error=False, fill_value=np.nan)
merged["x_enc_cm"] = interp_ex(merged["t_sec"])
merged["z_enc_cm"] = interp_ez(merged["t_sec"])

print(f"Merged rows        : {len(merged)}")
print(f"Camera valid rows  : {merged['x_cam_cm'].notna().sum()}")
print(f"Encoder valid rows : {merged['x_enc_cm'].notna().sum()}")
merged[["t_sec","x_cam_cm","z_cam_cm","x_enc_cm","z_enc_cm"]].head(5).round(3)


In [ ]:
# Cell 12 — Camera vs Encoder error stats
merged["err_x_cm"] = merged["x_enc_cm"] - merged["x_cam_cm"]
merged["err_z_cm"] = merged["z_enc_cm"] - merged["z_cam_cm"]
merged["err_2d_cm"]= np.sqrt(merged["err_x_cm"]**2 + merged["err_z_cm"]**2)

mask = merged["err_x_cm"].notna() & merged["err_z_cm"].notna()
ex = merged.loc[mask, "err_x_cm"].values
ez = merged.loc[mask, "err_z_cm"].values

def arr_stats(arr, name):
    print(f"{name:<20} mean={arr.mean():.3f}  std={arr.std():.3f}  "          f"min={arr.min():.3f}  max={arr.max():.3f}  "          f"RMSE={np.sqrt(np.mean(arr**2)):.3f}  "          f"95th={np.percentile(np.abs(arr),95):.3f}")

print(f"Valid frames : {mask.sum()} / {len(merged)}")
arr_stats(ex, "err_x_cm")
arr_stats(ez, "err_z_cm")
print(f"MAE X : {np.mean(np.abs(ex)):.3f} cm")
print(f"MAE Z : {np.mean(np.abs(ez)):.3f} cm")
print(f"Max |X|: {np.max(np.abs(ex)):.3f} cm")
print(f"Max |Z|: {np.max(np.abs(ez)):.3f} cm")


In [ ]:
# Cell 13 — Camera vs Encoder position plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle("NOARK Position: Camera vs Encoder", fontsize=14, fontweight="bold")

ax1.plot(merged["t_sec"], merged["x_cam_cm"], "k-",  lw=1.5, label="Camera X")
ax1.plot(merged["t_sec"], merged["x_enc_cm"], "r--", lw=1.2, label="Encoder X")
ax1.set_ylabel("X position (cm)"); ax1.legend(); ax1.grid(True, alpha=0.4)
ax1.set_title("X axis (left ↔ right)")

ax2.plot(merged["t_sec"], merged["z_cam_cm"], "k-",  lw=1.5, label="Camera Z")
ax2.plot(merged["t_sec"], merged["z_enc_cm"], color="darkorange", ls="--", lw=1.2, label="Encoder Z")
ax2.set_ylabel("Z position (cm)"); ax2.set_xlabel("Time (s)")
ax2.legend(); ax2.grid(True, alpha=0.4)
ax2.set_title("Z axis (up ↕ down)")

plt.tight_layout()
plt.savefig(RECORDING_DIR / "camera_vs_encoder_position.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Cell 14 — Error plot + box plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Camera vs Encoder Error", fontsize=13, fontweight="bold")

ax1 = axes[0]
ax1.plot(merged["t_sec"], merged["err_x_cm"], color="steelblue", lw=1, label="Error X")
ax1.plot(merged["t_sec"], merged["err_z_cm"], color="darkorange", lw=1, label="Error Z")
ax1.axhline(0, color="black", lw=0.8, ls="--")
ax1.set_xlabel("Time (s)"); ax1.set_ylabel("Error (cm)")
ax1.set_title("Encoder − Camera error over time")
ax1.legend(); ax1.grid(True, alpha=0.4)

ax2 = axes[1]
bp = ax2.boxplot([ex, ez], labels=["Error X", "Error Z"],
                 patch_artist=True, notch=False, widths=0.45,
                 medianprops=dict(color="red", linewidth=2),
                 flierprops=dict(marker="o", markersize=3, alpha=0.4))
for patch, color in zip(bp["boxes"], ["steelblue", "darkorange"]):
    patch.set_facecolor(color); patch.set_alpha(0.4)
ax2.axhline(0, color="grey", lw=0.8, ls="--")
ax2.set_ylabel("Error (cm)"); ax2.set_title("Error distribution")
ax2.grid(True, alpha=0.4, axis="y")

plt.tight_layout()
plt.savefig(RECORDING_DIR / "camera_vs_encoder_error.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Cell 15 — 2D trajectory in table frame
fig, ax = plt.subplots(figsize=(10, 8))
fig.suptitle("NOARK 2D Trajectory: Camera vs Encoder", fontsize=13, fontweight="bold")

valid = merged["x_cam_cm"].notna() & merged["x_enc_cm"].notna()
df_v  = merged[valid]

sc1 = ax.scatter(df_v["x_cam_cm"], df_v["z_cam_cm"],
                 c=df_v["t_sec"], cmap="Greys", s=8, alpha=0.8, label="Camera (GT)")
sc2 = ax.scatter(df_v["x_enc_cm"], df_v["z_enc_cm"],
                 c=df_v["t_sec"], cmap="Reds",  s=6, alpha=0.6, marker="x", label="Encoder")
plt.colorbar(sc1, ax=ax, label="Time (s)")

for name, pt in [("P2", P2), ("P4", P4)]:
    ax.plot(pt[0]*100, pt[2]*100, "k^", ms=10, zorder=5)
    ax.annotate(name, (pt[0]*100, pt[2]*100), xytext=(5,5),
                textcoords="offset points", fontsize=10, fontweight="bold")

ax.set_xlabel("X (cm)"); ax.set_ylabel("Z (cm)")
ax.set_title("X–Z plane  |  colour = time")
ax.legend(loc="lower right"); ax.grid(True, alpha=0.4)
ax.set_aspect("equal", adjustable="box")

plt.tight_layout()
plt.savefig(RECORDING_DIR / "camera_vs_encoder_trajectory.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Cell 16 — Save merged results to CSV
out_merged = RECORDING_DIR / "merged_results.csv"
merged.to_csv(out_merged, index=False)
print(f"Saved merged results → {out_merged}")
print(f"Columns : {merged.columns.tolist()}")
print(f"Shape   : {merged.shape}")
